# Phase 2B — GNINA CNN Rescoring (Top 50 Candidates)

## Purpose
CNN rescoring of top 50 Phase 2B candidates using GNINA's deep
learning scoring function trained on PDBbind. Provides more accurate
binding affinity estimates than Vina empirical scoring.

## Context
Phase 2B candidates were first screened with Smina (all 385 ligands,
1 receptor, CPU). The top 50 by Smina score are rescored here with
GNINA CNN across 3 representative ensemble receptors:
- 5AP7 (quality=9.6, holo)
- 5AP6 (quality=9.4, holo)
- 3CEK (quality=9.0, apo)

## Input files
- `top50_phase2b.zip` — top 50 Phase 2B PDBQT ligands
  (from `kinase_domain/data/phase2b/top50_pdbqt/`)
- `ensemble_receptors.zip` — 3 receptor PDBQTs
  (from `kinase_domain/data/receptor/ensemble/`)

## Output
- `top50_gnina_cnn_scores.csv` — CNN + Vina scores for all 50 candidates
  across 3 receptors

## To reproduce
1. Run `kinase_domain/scripts/phase2b_similarity_search.py`
2. Run `kinase_domain/scripts/prepare_ligands.py` for Phase 2B
3. Run `kinase_domain/scripts/run_smina_ensemble.py` locally
4. Copy top 50 PDBQTs to `kinase_domain/data/phase2b/top50_pdbqt/`
5. Upload zips to Colab and run this notebook

# Cell 1 - Setup

In [ ]:
from google.colab import drive, files
import os, zipfile, subprocess, time
import pandas as pd
from pathlib import Path

drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/mps1-drug-discovery'
os.makedirs('/content/top50', exist_ok=True)
os.makedirs('/content/receptors', exist_ok=True)
os.makedirs('/content/gnina_results', exist_ok=True)

# Install GNINA
!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina \
    -O /usr/local/bin/gnina
!chmod +x /usr/local/bin/gnina
!/usr/local/bin/gnina --version

Mounted at /content/drive
gnina  master:e9cb230+   Built Feb 11 2023.


# Cell 2 - Upload files

In [ ]:
# Upload top50_phase2b.zip and ensemble_receptors.zip
print("Upload top50_phase2b.zip and ensemble_receptors.zip")
uploaded = files.upload()

for fname in uploaded:
    if 'top50' in fname:
        with zipfile.ZipFile(fname) as z:
            z.extractall('/content/top50')
    elif 'receptor' in fname:
        with zipfile.ZipFile(fname) as z:
            z.extractall('/content/receptors')

ligands   = list(Path('/content/top50').rglob('*.pdbqt'))
receptors_all = list(Path('/content/receptors').rglob('*.pdbqt'))
receptors = [r for r in receptors_all
             if any(s in r.stem
                    for s in ['5AP7','5AP6','3CEK'])]

print(f"Ligands:   {len(ligands)}")
print(f"Receptors: {len(receptors)}")
for r in receptors:
    print(f"  {r.name}")

Upload top50_phase2b.zip and ensemble_receptors.zip


Saving ensemble_receptors.zip to ensemble_receptors.zip
Saving top50_phase2b.zip to top50_phase2b.zip
Ligands:   50
Receptors: 3
  3CEK_ensemble_receptor.pdbqt
  5AP6_ensemble_receptor.pdbqt
  5AP7_ensemble_receptor.pdbqt


# Cell 3 - GNINA CNN rescoring

In [ ]:
CENTER   = [-34.48, -15.66, -10.38]
BOX_SIZE = [20, 33, 21]
SELECTED = ['5AP7', '5AP6', '3CEK']

results = []
start   = time.time()

for lig_idx, lig_path in enumerate(sorted(ligands)):
    lig_name   = lig_path.stem
    rec_scores = {}
    rec_cnn    = {}

    for rec_path in sorted(receptors):
        rec_name = rec_path.stem.replace(
            '_ensemble_receptor', ''
        )
        out_path = (
            f"/content/gnina_results/"
            f"{lig_name}_{rec_name}_out.pdbqt"
        )

        cmd = [
            '/usr/local/bin/gnina',
            '--receptor',       str(rec_path),
            '--ligand',         str(lig_path),
            '--out',            out_path,
            '--center_x',       str(CENTER[0]),
            '--center_y',       str(CENTER[1]),
            '--center_z',       str(CENTER[2]),
            '--size_x',         str(BOX_SIZE[0]),
            '--size_y',         str(BOX_SIZE[1]),
            '--size_z',         str(BOX_SIZE[2]),
            '--exhaustiveness', '8',
            '--num_modes',      '5',
            '--cnn_scoring',    'rescore',
        ]

        try:
            r = subprocess.run(
                cmd, capture_output=True,
                text=True, timeout=180
            )
            vina_score = None
            cnn_score  = None
            for line in r.stdout.split('\n'):
                parts = line.split()
                if len(parts) >= 3 and parts[0] == '1':
                    try:
                        vina_score = float(parts[1])
                        cnn_score  = float(parts[2])
                        break
                    except:
                        pass
            rec_scores[rec_name] = vina_score
            rec_cnn[rec_name]    = cnn_score
        except Exception as e:
            rec_scores[rec_name] = None
            rec_cnn[rec_name]    = None

    valid_vina = [s for s in rec_scores.values()
                  if s is not None]
    valid_cnn  = [s for s in rec_cnn.values()
                  if s is not None]

    if valid_vina:
        result = {
            'ligand':       lig_name,
            'best_vina':    round(min(valid_vina), 3),
            'mean_vina':    round(sum(valid_vina)/
                            len(valid_vina), 3),
            'best_cnn':     round(min(valid_cnn), 3)
                            if valid_cnn else None,
            'mean_cnn':     round(sum(valid_cnn)/
                            len(valid_cnn), 3)
                            if valid_cnn else None,
            'n_receptors':  len(valid_vina),
            'consensus':    sum(1 for s in valid_vina
                                if s < -7.0) >= 2,
        }
        for rec in SELECTED:
            result[f'vina_{rec}'] = rec_scores.get(rec)
            result[f'cnn_{rec}']  = rec_cnn.get(rec)
        results.append(result)

        elapsed = time.time() - start
        eta     = (elapsed/(lig_idx+1)) * (
            len(ligands)-lig_idx-1
        )
        print(f"[{lig_idx+1}/{len(ligands)}] "
              f"{lig_name[:35]:<35} "
              f"vina={result['best_vina']:.3f} "
              f"cnn={result['best_cnn']:.3f} "
              f"ETA: {eta/60:.0f} min")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_cnn')
results_df.to_csv(
    f"{PROJECT}/top50_gnina_cnn_scores.csv",
    index=False
)
print(f"\nDone! Saved → {PROJECT}/top50_gnina_cnn_scores.csv")
print(f"\nTop 10 by CNN score:")
for _, r in results_df.head(10).iterrows():
    print(f"  {r['ligand'][:40]:<40} "
          f"CNN={r['best_cnn']:.3f} "
          f"Vina={r['best_vina']:.3f}")

[1/50] phase2b_seed_0_24872560             vina=-8.300 cnn=0.448 ETA: 187 min
[2/50] phase2b_seed_0_49843471             vina=-9.570 cnn=0.336 ETA: 166 min
[3/50] phase2b_seed_0_5291                 vina=-8.920 cnn=0.666 ETA: 204 min
[4/50] phase2b_seed_0_5330286              vina=-10.590 cnn=0.276 ETA: 196 min
[5/50] phase2b_seed_0_60195662             vina=-7.440 cnn=0.279 ETA: 200 min
[6/50] phase2b_seed_0_68029831             vina=-10.700 cnn=0.880 ETA: 185 min
[7/50] phase2b_seed_0_9919680              vina=-10.090 cnn=0.376 ETA: 177 min
[8/50] phase2b_seed_0_9939865              vina=-8.890 cnn=0.265 ETA: 176 min
[9/50] phase2b_seed_1_10433653             vina=-9.610 cnn=0.627 ETA: 175 min
[10/50] phase2b_seed_1_11338127             vina=-8.770 cnn=0.419 ETA: 167 min
[11/50] phase2b_seed_1_11373432             vina=-9.040 cnn=0.454 ETA: 159 min
[12/50] phase2b_seed_1_11494412             vina=-10.020 cnn=0.446 ETA: 151 min
[13/50] phase2b_seed_1_11618268             vina=-7.860 c